# LAMPS CrewAI Full Live Pipeline on PyPI Packages

Runs the paper-style live pipeline through CrewAI itself:

`Fetcher Agent -> Extractor Agent -> Classifier Agent -> Verdict Agent`

Backends:
- Fetcher Agent: CrewAI + PyPI JSON/source archive download tool.
- Extractor Agent: CrewAI + Ollama Cloud semantic file selection over full package archives.
- Classifier Agent: fine-tuned CodeBERT, per-file malicious/benign.
- Verdict Agent: CrewAI + Ollama Cloud conservative package verdict and rationale.

Drive requirements:
- `NT230/data/d1/saved_models/checkpoint-best-acc/model.bin`
- optional package list: `NT230/data/live_packages.txt`
- Ollama key in `NT230/data/.env` as `$env:OLLAMA_API_KEY = "..."`, or set it manually in Colab.

Do not `pip install` analyzed packages. This notebook downloads source archives only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, re, sys, json, shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/My Drive/NT230')
DRIVE_DATA = DRIVE_ROOT / 'data'
MODEL_PATH = DRIVE_DATA / 'd1/saved_models/checkpoint-best-acc/model.bin'
PACKAGE_LIST_PATH = DRIVE_DATA / 'live_packages.txt'
OUTPUT_DIR = DRIVE_DATA / 'live_crewai_results'
LOCAL_MODEL = Path('/content/saved_models/checkpoint-best-acc/model.bin')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL.parent.mkdir(parents=True, exist_ok=True)
assert MODEL_PATH.exists(), f'Missing model.bin: {MODEL_PATH}'
shutil.copy(MODEL_PATH, LOCAL_MODEL)
print('Model:', LOCAL_MODEL, round(LOCAL_MODEL.stat().st_size / 1e6, 1), 'MB')
print('Package list:', PACKAGE_LIST_PATH if PACKAGE_LIST_PATH.exists() else 'default list')
print('Output:', OUTPUT_DIR)


In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm "crewai[tools]" litellm ollama openai


In [ ]:
REPO_DIR = Path('/content/NT230')
if not REPO_DIR.exists():
    !git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
else:
    !git -C /content/NT230 fetch origin main
    !git -C /content/NT230 reset --hard origin/main
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready:', REPO_DIR)
!git -C /content/NT230 log --oneline -1


In [ ]:
# Load OLLAMA_API_KEY from Drive .env if present.
env_path = DRIVE_DATA / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        m = re.match(r'^\$env:(\w+)\s*=\s*["\']?([^"\']+)["\']?', line)
        if m:
            os.environ.setdefault(m.group(1), m.group(2).strip())
        elif '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

os.environ.setdefault('OLLAMA_MODEL', 'deepseek-v4-flash:cloud')
os.environ.setdefault('OLLAMA_HOST', 'https://ollama.com')
# CrewAI uses LiteLLM. Use Ollama chat provider route for tool-using agents.
os.environ.setdefault('CREWAI_OLLAMA_MODEL', f"ollama_chat/{os.environ['OLLAMA_MODEL']}")

print('OLLAMA_API_KEY:', 'SET' if os.getenv('OLLAMA_API_KEY') else 'NOT FOUND')
print('OLLAMA_MODEL:', os.getenv('OLLAMA_MODEL'))
print('OLLAMA_HOST:', os.getenv('OLLAMA_HOST'))
print('CREWAI_OLLAMA_MODEL:', os.getenv('CREWAI_OLLAMA_MODEL'))

# Optional manual fallback:
# os.environ['OLLAMA_API_KEY'] = 'your-key-here'


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')


In [ ]:
from crewai import LLM
from lamps.agents.classifier import ClassifierAgent
from lamps.agents.extractor import LLMArchiveExtractorAgent
from lamps.agents.fetcher import FetcherAgent
from lamps.agents.verdict import VerdictAgent
from lamps.crewai_pipeline import CrewAIExecution, CrewAIStep, LampsCrewPipeline, LampsCrewResult
from lamps.llms.ollama_client import OllamaClient

assert os.getenv('OLLAMA_API_KEY'), 'Missing OLLAMA_API_KEY. Put it in NT230/data/.env or set it manually.'

OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'deepseek-v4-flash:cloud')
OLLAMA_HOST = os.getenv('OLLAMA_HOST', 'https://ollama.com')
CREWAI_OLLAMA_MODEL = os.getenv('CREWAI_OLLAMA_MODEL', f'ollama_chat/{OLLAMA_MODEL}')

# Native Ollama client: used inside Extractor/Verdict tools for paper reasoning prompts.
reasoning_llm = OllamaClient(
    model=OLLAMA_MODEL,
    host=OLLAMA_HOST,
)

# CrewAI LLM: use LiteLLM's Ollama chat provider, not OpenAI provider.
# Wrong route caused: ValueError('Invalid response from LLM call - None or empty.').
crew_llm = LLM(
    model=CREWAI_OLLAMA_MODEL,
    api_key=os.environ['OLLAMA_API_KEY'],
    base_url=OLLAMA_HOST,
    temperature=0,
)

pipeline = LampsCrewPipeline(
    fetcher=FetcherAgent(),
    extractor=LLMArchiveExtractorAgent(llm=reasoning_llm),
    classifier=ClassifierAgent(checkpoint=str(LOCAL_MODEL), batch_size=64),
    verdict=VerdictAgent(llm=reasoning_llm),
    crew_llm=crew_llm,
    verbose=False,
)
pipeline.crew = pipeline.build_crew()
print('CrewAI agents:', [a.role for a in pipeline.crew.agents])
print('CrewAI tasks:', len(pipeline.crew.tasks))
print('CrewAI LLM:', CREWAI_OLLAMA_MODEL, '@', OLLAMA_HOST)

def _files_summary(files):
    return [{'path': f.rel_path, 'chars': len(f.source)} for f in files]

def _classification_summary(classifications):
    return [
        {'path': c.rel_path, 'label': c.label, 'target': c.target, 'score': c.score}
        for c in classifications
    ]

def run_paper_crewai(package: str, version: str | None = None):
    pipeline._state = {}
    crew_output = pipeline.crew.kickoff(inputs={'package': package, 'version': version or ''})

    fetch = pipeline._state.get('fetch')
    files = pipeline._state.get('files')
    classifications = pipeline._state.get('classifications')
    verdict = pipeline._state.get('verdict')
    missing = [
        name for name, value in {
            'fetch': fetch,
            'files': files,
            'classifications': classifications,
            'verdict': verdict,
        }.items() if value is None
    ]
    if missing:
        raise RuntimeError(f'CrewAI did not complete required tool calls: {missing}. Raw output: {crew_output}')

    steps = [
        CrewAIStep('Fetcher Agent', 'fetch', {
            'package': fetch.package,
            'version': fetch.version,
            'archive_path': str(fetch.archive_path),
            'archive_url': fetch.archive_url,
        }),
        CrewAIStep('Extractor Agent', 'extract', {
            'n_files': len(files),
            'files': _files_summary(files),
        }),
        CrewAIStep('Classifier Agent', 'classify', {
            'n_files': len(classifications),
            'n_malicious': sum(1 for c in classifications if c.target == 1),
            'predictions': _classification_summary(classifications),
        }),
        CrewAIStep('Verdict Agent', 'aggregate', verdict.to_dict()),
    ]
    pipeline.last_execution = CrewAIExecution(package=package, steps=steps)
    return LampsCrewResult(package=package, verdict=verdict, fetch=fetch, files=classifications), str(crew_output)


In [ ]:
def parse_package_line(line: str):
    line = line.strip()
    if not line or line.startswith('#'):
        return None
    if '==' in line:
        package, version = line.split('==', 1)
        return package.strip(), version.strip()
    return line, None

if PACKAGE_LIST_PATH.exists():
    package_specs = [parse_package_line(x) for x in PACKAGE_LIST_PATH.read_text(encoding='utf-8').splitlines()]
    package_specs = [x for x in package_specs if x]
else:
    package_specs = [
        ('requests', None),
        ('urllib3', None),
        ('certifi', None),
        ('spacestudio-orbit-propagation', '1.1.3'),
        ('grandslam', '0.1.0'),
    ]

print('Packages to analyze:', len(package_specs))
print(package_specs[:20])


In [ ]:
from tqdm import tqdm

results = []
errors = []
out_jsonl = OUTPUT_DIR / 'package_results.jsonl'
out_errors = OUTPUT_DIR / 'errors.jsonl'
out_summary = OUTPUT_DIR / 'summary.json'
for path in (out_jsonl, out_errors, out_summary):
    if path.exists():
        path.unlink()

for package, version in tqdm(package_specs, desc='LAMPS CrewAI live'):
    try:
        result, crew_raw_output = run_paper_crewai(package, version=version)
        payload = result.to_dict()
        payload['crew_execution'] = pipeline.last_execution.to_dict() if pipeline.last_execution else None
        payload['crew_raw_output'] = crew_raw_output
        results.append(payload)
        with out_jsonl.open('a', encoding='utf-8') as f:
            f.write(json.dumps(payload, ensure_ascii=False) + '\n')
        print(package, '=>', payload['verdict']['label'], 'files=', payload['verdict']['n_files'])
    except Exception as exc:
        err = {'package': package, 'version': version, 'error': repr(exc)}
        errors.append(err)
        with out_errors.open('a', encoding='utf-8') as f:
            f.write(json.dumps(err, ensure_ascii=False) + '\n')
        print('ERROR', package, repr(exc))

summary = {
    'pipeline': 'CrewAI kickoff: Fetcher -> Extractor -> Classifier -> Verdict',
    'ollama_model': OLLAMA_MODEL,
    'crew_llm_model': CREWAI_OLLAMA_MODEL,
    'ollama_host': OLLAMA_HOST,
    'n_requested': len(package_specs),
    'n_success': len(results),
    'n_errors': len(errors),
    'results_path': str(out_jsonl),
    'errors_path': str(out_errors),
}
out_summary.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


In [ ]:
# Quick view
for payload in results[:10]:
    verdict = payload['verdict']
    print(payload['package'], verdict['label'], 'target=', verdict['target'], 'n_files=', verdict['n_files'])
